## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv()

True

In [4]:
model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    temperature=0.7,
)

response = model.invoke("What is RAG?")
response

AIMessage(content='RAG stands for **Retrieval-Augmented Generation**. It\'s a powerful technique that combines the strengths of two key AI components:\n\n1.  **Retrieval:** This part involves searching and fetching relevant information from a large knowledge base (like a database, a collection of documents, or the internet). Think of it as a smart search engine.\n2.  **Generation:** This part uses a large language model (LLM) to create new text based on the retrieved information and a given prompt.\n\n**In essence, RAG aims to improve the quality and accuracy of LLM-generated text by grounding it in factual, up-to-date, and specific information.**\n\nHere\'s a breakdown of how it works and why it\'s important:\n\n**How RAG Works:**\n\n1.  **User Query:** A user asks a question or provides a prompt.\n2.  **Retrieval:** The RAG system first uses the user\'s query to search a pre-defined knowledge base. This knowledge base can be anything from internal company documents, a specific set of

In [8]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather in a city"""
    return f"The weather in {city} today is sunny wit a temperature of 75°F"

model_tools = model.bind_tools([get_weather])

response = model_tools.invoke("What is the weather in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool call: {tool_call['name']}, arguments: {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "New York"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019de8e7-766b-7651-bab6-2a12be7d6a7f-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'e55f61fe-b754-4af3-a723-aeb1ec3ff85c', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 16, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}}
Tool call: get_weather, arguments: {'city': 'New York'}


### Tool execution loop

In [11]:
# Step 1: Model returns tool calls
messages = [
    {"role": "user", "content": "What is the weather in Dallas?"},
]

response = model_tools.invoke(messages)
messages.append(response)

# Step 2: Implement tool execution loop
tools_by_name = {"get_weather": get_weather}

for tool_call in response.tool_calls:
    tool_name = tool_call["name"]
    print(tool_name)
    tool_result = tools_by_name[tool_name].invoke(tool_call)
    messages.append(tool_result)

# Step 3: Return final answer
final_response = model_tools.invoke(messages)
final_response.content

get_weather


'The weather in Dallas today is sunny with a temperature of 75°F.'

In [12]:
messages

[{'role': 'user', 'content': 'What is the weather in Dallas?'},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Dallas"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019de8ed-a9a9-7072-ac07-db1c2aa8f7b7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Dallas'}, 'id': '9d7a528c-865f-4d03-a41d-90cfae6910de', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 15, 'total_tokens': 62, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='The weather in Dallas today is sunny wit a temperature of 75°F', name='get_weather', tool_call_id='9d7a528c-865f-4d03-a41d-90cfae6910de')]